In [36]:
import numpy as np
import pandas as pd 

import matplotlib.pyplot as plt

import h5py
from tqdm import tqdm

%matplotlib inline

### Preparing the Atmospheric parameters and MK spectral classes

In [2]:
data_path = "/home/arbiter/projects/Survey-invariant-generalization/data/usable_data"

In [3]:
import os

In [4]:
data_path_list = os.listdir(data_path)
data_path_list

['lamost_meta.csv',
 'sdss_resampled.h5',
 'lamost_resampled.h5',
 'sdss_meta.csv']

In [5]:
sdss_data = h5py.File(f"{data_path}/sdss_resampled.h5")
len(sdss_data.keys())

21960

In [8]:
sdss_meta_data = pd.read_csv(f"{data_path}/sdss_meta.csv")
sdss_meta_data.shape

(21998, 13)

In [9]:
sdss_meta_data.head()

,Unnamed: 0,specobjid,ra,dec,plate,mjd,fiberid,subclass,z,elodieTeff,elodieLogG,elodieFeH,No.
0,0,1.005493e+18,123.77894,37.339432,893,52589,234,M3,0.000014,3717,4.890,-0.20,NaN
1,1,1.005500e+18,123.87184,37.228072,893,52589,260,M0,0.000107,3858,4.165,-0.06,NaN
2,2,1.005503e+18,123.62071,37.652263,893,52589,272,M3,-0.000056,3717,4.890,-0.20,NaN
3,3,1.658451e+18,328.65518,-1.165836,1473,52908,3,M1,-0.000229,3980,4.958,-0.04,NaN
4,4,1.658457e+18,329.05391,-0.489388,1473,52908,23,M1,-0.000148,3980,4.958,-0.04,NaN


In [15]:
star_id = "1000-198-52643"
star = sdss_data[star_id]

In [25]:
star_id_info = star_id.split("-")
star_id_info

['1000', '198', '52643']

In [28]:
star_meta = sdss_meta_data.loc[
    (sdss_meta_data["plate"] == int(star_id_info[0])) &
    (sdss_meta_data["fiberid"] == int(star_id_info[1])) &
    (sdss_meta_data["mjd"] == int(star_id_info[2]))
]

In [43]:
star_meta["elodieFeH"].iloc[0]

np.float64(-1.5)

In [44]:
star_meta["elodieLogG"].iloc[0]

np.float64(4.0)

In [45]:
star_meta["elodieTeff"].iloc[0]

np.int64(8500)

In [33]:
star_meta["subclass"].iloc[0][0]

'A'

In [48]:
def get_flux_labels(data, meta_data):
    
    flux_list     = []
    teff_list     = []
    logg_list     = []
    feh_list      = []
    cls_list      = []
    failed        = 0

    # Map spectral class letter to integer index
    class_map = {"O": 0, "B": 1, "A": 2, "F": 3, "G": 4, "K": 5, "M": 6}

    for star_id in tqdm(data):
        star_id_info = star_id.split("-")

        try:
            # Flux
            flux_array = data[star_id]["flux"][:]

            # Metadata lookup
            star_meta = meta_data.loc[
                (meta_data["plate"]   == int(star_id_info[0])) &
                (meta_data["fiberid"] == int(star_id_info[1])) &
                (meta_data["mjd"]     == int(star_id_info[2]))
            ]

            # Skip if no match found
            if len(star_meta) == 0:
                failed += 1
                continue

            teff = star_meta["elodieTeff"].iloc[0]
            logg = star_meta["elodieLogG"].iloc[0]
            feh  = star_meta["elodieFeH"].iloc[0]
            spectral_class = star_meta["subclass"].iloc[0][0]

            # Skip if any label is null
            if pd.isna(teff) or pd.isna(logg) or pd.isna(feh):
                failed += 1
                continue

            # Skip if class not in map
            if spectral_class not in class_map:
                failed += 1
                continue

            flux_list.append(flux_array)
            teff_list.append(teff)
            logg_list.append(logg)
            feh_list.append(feh)
            cls_list.append(class_map[spectral_class])

        except Exception as e:
            print(f"Failed for {star_id}: {e}")
            failed += 1
            continue

    print(f"Collected: {len(flux_list)} | Failed/skipped: {failed}")

    # Stack into arrays
    flux_array  = np.array(flux_list)                          # (n_stars, 3800)
    y_reg       = np.column_stack([teff_list, logg_list, feh_list])  # (n_stars, 3)
    y_cls       = np.array(cls_list)                           # (n_stars,)

    return flux_array, y_reg, y_cls

In [49]:
flux, y_reg, y_cls = get_flux_labels(sdss_data, sdss_meta_data)

100%|██████████| 21960/21960 [00:20<00:00, 1092.28it/s]


Collected: 21960 | Failed/skipped: 0


In [50]:
y_cls[0]

np.int64(2)

In [51]:
y_reg[0]

array([ 8.5e+03,  4.0e+00, -1.5e+00])